In [1]:
import pandas as pd
import numpy as np
from itertools import combinations

### Read the dataset - Excel Version

In [2]:
df = pd.read_excel("cn_dataset_styles - Excel Version.xlsx")

### General function of allocation for specific base_claim_id - extra ones too

In [15]:
def allocation(df, base_claim_id):
    narrative_ids = list(df[df['base_claim_id'] == base_claim_id]['narrative_id'])
    combs = list(combinations(narrative_ids, 2))
    np.random.shuffle(combs)
    columns = ['left_narrative_id', 'right_narrative_id']
    for i in range(0,10,2):
        df_eval = pd.DataFrame(combs[i*170: (i+1)*170], columns=columns)
        df_extra = pd.DataFrame(combs[(i+1)*170: (i+2)*170], columns=columns)
        df_eval['base_claim_id'] = base_claim_id
        df_extra['base_claim_id'] = base_claim_id
        expanded_df = df_eval.loc[df_eval.index.repeat(3)].reset_index(drop=True)
        expanded_df['kpi_id'] = [1, 2, 3] * (len(expanded_df) // 3)
        expanded_extra_df = df_extra.loc[df_extra.index.repeat(3)].reset_index(drop=True)
        expanded_extra_df['kpi_id'] = [1, 2, 3] * (len(expanded_extra_df) // 3)
        with pd.ExcelWriter(f"evaluator_{i//2+1}.xlsx", mode='a', engine='openpyxl', if_sheet_exists='overlay') as writer:
            expanded_df.to_excel(writer, sheet_name='Sheet1', index=False, header=False, startrow=writer.sheets['Sheet1'].max_row)
        with pd.ExcelWriter(f"extra_evaluator_{i//2+1}.xlsx", mode='a', engine='openpyxl', if_sheet_exists='overlay') as writer:
            expanded_extra_df.to_excel(writer, sheet_name='Sheet1', index=False, header=False, startrow=writer.sheets['Sheet1'].max_row)

### Using the function we just built

In [16]:
for i in range(np.max(df['base_claim_id'])):
    allocation(df, i+1)